# 01. The memory model and mutability

**Goal.** Predict what Python does with memory when you assign, copy, mutate and delete.

**You will be able to:**
1. Explain why a name is a pointer and an assignment copies a pointer, not an object.
2. Choose between immutable and mutable types, and between shallow and deep copies.
3. Describe reference counting, the cyclic collector and when each frees memory.
4. Read bytecode and estimate object sizes.

Every section follows one pattern: how it works, where it runs in production, a trade-off table, then code with printed evidence.

## 1. Names, objects and identity

### How it works under the hood
- Each CPython object starts with a `PyObject` header: a reference count and a pointer to its type. The payload follows.
- A name is a key in a namespace dictionary (a function's fast locals, a module's `globals()`). It maps to the object's address.
- `b = a` copies one pointer and raises the object's reference count by one. No data moves.
- `id(x)` returns that address in CPython. `x is y` compares addresses. `x == y` calls `type(x).__eq__`.

### Production use
- A shared settings `dict` mutated by one request handler changes behaviour for every other handler.
- Passing a large DataFrame or list to a function costs one pointer, so mutating it there changes the caller's data.

### Trade-offs

| Model | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Reference semantics (Python) | One pointer per name | O(1) assignment, any size | Simple until aliasing surprises | Cheap passing, easy sharing | Hidden coupling through aliases |
| Value semantics (C struct copy) | Full copy per assignment | O(size) assignment | Explicit, no aliasing | Isolation by default | Copy cost, manual pointers for sharing |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
orders = [{"id": 1001, "total": 42.5}, {"id": 1002, "total": 9.0}]
alias = orders
snapshot = list(orders)

orders.append({"id": 1003, "total": 120.0})
print("alias sees append     :", len(alias), "orders")
print("shallow copy does not :", len(snapshot), "orders")
print("alias is orders       :", alias is orders, "| same id:", id(alias) == id(orders))

snapshot[0]["total"] = 0.0
print("inner dict is shared  :", orders[0]["total"])


def apply_discount(order_list, pct):
    for o in order_list:
        o["total"] = round(o["total"] * (1 - pct / 100), 2)


apply_discount(orders[1:], 10)
print("callee mutated caller :", [o["total"] for o in orders])

x = [1, 2]
before = id(x)
x += [3]
print("list += mutates in place:", id(x) == before)
t = (1, 2)
before = id(t)
t += (3,)
print("tuple += rebinds a new object:", id(t) != before, t)

alias sees append     : 3 orders
shallow copy does not : 2 orders
alias is orders       : True | same id: True
inner dict is shared  : 0.0
callee mutated caller : [0.0, 8.1, 108.0]
list += mutates in place: True
tuple += rebinds a new object: True (1, 2, 3)


## 2. Mutability, copying and shared defaults

### How it works under the hood
- Immutable types (`int`, `str`, `bytes`, `tuple`, `frozenset`) never change after creation. CPython can therefore hash, share and intern them safely.
- A shallow copy (`list(x)`, `x.copy()`, `copy.copy`) allocates a new container and copies the pointers. Children stay shared.
- `copy.deepcopy` walks the object graph and keeps a memo dictionary keyed by `id`, so cycles and shared children copy once.
- Default argument values are evaluated once, when `def` runs. They live in `func.__defaults__`, so a mutable default is shared by every call.

### Production use
- Build request-scoped state with `None` sentinels or `dataclasses.field(default_factory=...)`, never with a literal `[]`.
- Return `tuple` or `frozenset` from public APIs so callers cannot corrupt internal state.
- Deep-copy configuration before per-tenant overrides.

### Trade-offs

| Choice | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Immutable value (`tuple`) | Compact, hashable | Fast creation, no defensive copy | Clear intent | Safe sharing, usable as dict key | New object per change |
| Mutable value (`list`) | Over-allocated | Fast in-place edits | Familiar | Cheap updates | Aliasing bugs |
| Shallow copy | One new container | O(n) pointers | Easy to miss shared children | Fast | Nested mutation leaks |
| Deep copy | Whole graph duplicated | O(graph) plus memo cost | Safe and explicit | Full isolation | Slow, can break on exotic objects |

In [3]:
import copy
import timeit

order = {"id": 7, "customer": "ada", "lines": [{"sku": "pen", "qty": 3}, {"sku": "pad", "qty": 1}]}
shallow, deep = copy.copy(order), copy.deepcopy(order)
order["lines"][0]["qty"] = 99
print("shallow sees edit:", shallow["lines"][0]["qty"], "| deep isolated:", deep["lines"][0]["qty"])

shallow_t = timeit.timeit(lambda: copy.copy(order), number=20_000)
deep_t = timeit.timeit(lambda: copy.deepcopy(order), number=20_000)
print(f"copy.copy {shallow_t * 50:.2f} us | deepcopy {deep_t * 50:.2f} us | ratio {deep_t / shallow_t:.0f}x")


def add_tag_buggy(tag, tags=[]):
    tags.append(tag)
    return tags


def add_tag(tag, tags=None):
    tags = [] if tags is None else tags
    tags.append(tag)
    return tags


print("buggy:", add_tag_buggy("a"), add_tag_buggy("b"))
print("stored default:", add_tag_buggy.__defaults__)
print("fixed:", add_tag("a"), add_tag("b"))

pair = ("config", [1, 2])
try:
    pair[1] += [3]
except TypeError as exc:
    print("tuple item assignment failed:", exc)
print("yet the list inside changed:", pair)

shallow sees edit: 99 | deep isolated: 3
copy.copy 0.19 us | deepcopy 8.29 us | ratio 43x
buggy: ['a', 'b'] ['a', 'b']
stored default: (['a', 'b'],)
fixed: ['a'] ['b']
tuple item assignment failed: 'tuple' object does not support item assignment
yet the list inside changed: ('config', [1, 2, 3])


## 3. Reference counting and the cyclic collector

### How it works under the hood
- Every new reference increments `ob_refcnt`. Every dropped reference decrements it. At zero CPython frees the object at once and runs `__del__`.
- Reference counting cannot free a cycle: `a -> b -> a` keeps both counts above zero.
- The cyclic collector tracks container objects in generations. It finds groups reachable only from each other and frees them. `gc.get_threshold()` controls when each generation runs.
- `weakref` references do not raise the count. A `WeakValueDictionary` entry disappears when its value dies.

### Production use
- Long-running workers leak memory when caches hold strong references. Use `weakref` or a bounded cache.
- Some web fleets freeze or disable the cyclic collector in forked workers to keep pages shared copy-on-write. They accept manual `gc.collect()` calls.
- Deterministic release makes `with` blocks and file handles predictable in CPython. PyPy and other runtimes do not guarantee this.

### Trade-offs

| Strategy | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Reference counting | Counter per object (8 bytes) | Tiny pauses spread over every assignment | Invisible to the programmer | Immediate, predictable release | Cannot free cycles, adds cost to every reference change |
| Cyclic collector | Extra tracking headers | Occasional pauses, longer for old generations | Invisible until tuned | Frees cycles | Non-deterministic timing |
| Manual (C, Rust ownership) | Minimal | None at runtime | Needs discipline or a borrow checker | Full control | Leaks and use-after-free in C |

In [4]:
import gc
import weakref

from understanding_python.internals import Tracked, collect, make_cycle, refcount

blob = object()
base = refcount(blob)
holder = [blob, blob]
print("extra references from a list holding it twice:", refcount(blob) - base)
del holder
print("extra references after del holder            :", refcount(blob) - base)

Tracked.reclaimed.clear()
t = Tracked("plain")
del t
print("plain object freed at once:", Tracked.reclaimed)

make_cycle("loop")
print("cycle freed yet?          :", "loop-a" in Tracked.reclaimed)
print("collector found", collect(), "unreachable objects")
print("cycle freed after collect :", sorted(n for n in Tracked.reclaimed if n.startswith("loop")))
print("generation thresholds     :", gc.get_threshold())


class Session:
    def __init__(self, user):
        self.user = user


live = weakref.WeakValueDictionary()
s = Session("ada")
live["ada"] = s
print("cache holds session:", "ada" in live)
del s
print("after last strong reference dropped:", "ada" in live)

extra references from a list holding it twice: 2
extra references after del holder            : 0
plain object freed at once: ['plain']
cycle freed yet?          : False


collector found 2 unreachable objects
cycle freed after collect : ['loop-a', 'loop-b']
generation thresholds     : (2000, 10, 10)
cache holds session: True
after last strong reference dropped: False


## 4. Interning, object sizes and bytecode

### How it works under the hood
- CPython preallocates the integers -5 to 256. Equal small values share one object. Larger integers are new objects unless the compiler folds constants.
- The compiler interns identifier-like string constants. `sys.intern` interns any string at run time, so equal keys share one object and compare by pointer first.
- `sys.getsizeof` reports the shallow size. A small `int` takes 28 bytes. A list stores 8-byte pointers and over-allocates so `append` stays amortised O(1).
- The compiler turns each function into bytecode for a stack machine. The interpreter loop dispatches one opcode at a time. Since 3.11 it rewrites hot opcodes into specialised forms such as `BINARY_OP_ADD_INT`.

### Production use
- Interning repeated keys in a million-row parse cuts memory and speeds dictionary lookups.
- Use `is` only for `None`, `True`, `False` and sentinels. Use `==` for values.
- Read `dis` output to see why a loop is slow. Local variable access (`LOAD_FAST`) is cheaper than global access.

### Trade-offs

| Technique | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| `sys.intern` on repeated strings | Saves one copy per duplicate | Faster equality and hashing | One extra call | Large savings on repeated tokens | Interned table lives long, wasted on unique strings |
| Small-int cache | None to spare | No allocation for -5 to 256 | Invisible | Free speed | Invites wrong use of `is` |
| Reading bytecode | None | None | Needs practice | Explains real costs | Opcodes change between versions |

In [5]:
import dis
import sys

from understanding_python.internals import deep_size, opnames

a, b = int("256"), int("256")
c, d = int("257"), int("257")
print("256 shared:", a is b, "| 257 shared:", c is d, "| 257 equal:", c == d)

words = ["".join(["status=", "ok"]) for _ in range(3)]
pooled = [sys.intern(w) for w in words]
print("distinct ids before intern:", len({id(w) for w in words}), "| after:", len({id(w) for w in pooled}))

for obj in (0, 2**30, 2**60, "", "abc", (), (1, 2, 3), [], [1, 2, 3], {}, {"a": 1}):
    print(f"{type(obj).__name__:6} {sys.getsizeof(obj):4d} bytes  {obj!r:.18}")

sizes = []
grow = []
for i in range(40):
    grow.append(i)
    if not sizes or sys.getsizeof(grow) != sizes[-1][1]:
        sizes.append((len(grow), sys.getsizeof(grow)))
print("list capacity jumps (len, bytes):", sizes)

record = {"user": "ada", "tags": ["x" * 50, "y" * 50], "scores": list(range(100))}
print("shallow", sys.getsizeof(record), "bytes | deep", deep_size(record), "bytes")


def total_with_tax(price, rate):
    return price + price * rate


print(opnames(total_with_tax)[:12])
dis.dis(total_with_tax)

for _ in range(2000):
    total_with_tax(10.0, 0.2)
print("--- after warm-up, specialised ---")
dis.dis(total_with_tax, adaptive=True)

256 shared: True | 257 shared: False | 257 equal: True
distinct ids before intern: 3 | after: 1
int      28 bytes  0
int      32 bytes  1073741824
int      36 bytes  115292150460684697
str      41 bytes  ''
str      44 bytes  'abc'
tuple    40 bytes  ()
tuple    64 bytes  (1, 2, 3)
list     56 bytes  []
list     88 bytes  [1, 2, 3]
dict     64 bytes  {}
dict    184 bytes  {'a': 1}
list capacity jumps (len, bytes): [(1, 88), (5, 120), (9, 184), (17, 248), (25, 312), (33, 376)]
shallow 184 bytes | deep 4275 bytes
['RESUME', 'LOAD_FAST_LOAD_FAST', 'LOAD_FAST', 'BINARY_OP', 'BINARY_OP', 'RETURN_VALUE']
 29           RESUME                   0

 30           LOAD_FAST_LOAD_FAST      0 (price, price)
              LOAD_FAST                1 (rate)
              BINARY_OP                5 (*)
              BINARY_OP                0 (+)
              RETURN_VALUE
--- after warm-up, specialised ---
 29           RESUME_CHECK             0

 30           LOAD_FAST_LOAD_FAST      0 (price, price